### Use virtual environment "guidelinerag"

In [11]:
from dotenv import load_dotenv
from openai import AzureOpenAI

import base64
import json
import os
import requests
import time
import pandas as pd
from tqdm import tqdm
import re
import grobid_tei_xml
from langchain_text_splitters import RecursiveCharacterTextSplitter

ENV_PATH = os.path.join(os.getcwd(), '..', '.env')
load_dotenv(dotenv_path=ENV_PATH);

In [12]:
from langchain_text_splitters import MarkdownTextSplitter
from langchain_core.documents import Document

In [13]:
root_dir= os.path.expanduser("~/Documents/QPain_clean")
output_dir= os.path.join(root_dir, "guideline-RAG", "guideline-corpus", "perioperative-pain-guideline-chunks")

In [14]:
periop_guideline_dir = os.path.join(root_dir, "guideline-RAG", "guideline-corpus", "PubMed_periop_painmanagement", "perioperative_care", "str_text_files")
painmg_guideline_dir = os.path.join(root_dir, "guideline-RAG", "guideline-corpus", "PubMed_periop_painmanagement", "pain_management", "str_text_files")
pdf_guideline_dir = os.path.join(root_dir, "guideline-RAG", "guideline-corpus", "PubMed_periop_painmanagement", "pdf_guidelines")

In [15]:
import json
from collections.abc import Iterable
from langchain_core.documents import Document

def save_docs_to_jsonl(array: Iterable[Document], file_path: str) -> None:
    """
    Saves a list of LangChain Documents to a JSONL file.
    """
    with open(file_path, 'w') as jsonl_file:
        for doc in array:
            jsonl_file.write(json.dumps(doc.model_dump()) + '\n')

# Layout-aware Chunking Complete

In [16]:
def layout_aware_chunking(guideline_dir: str, output_file_name: str, max_chunk_size: int = 1000) -> list[Document]:
    # Iterate through the clinical guideline files in the specified directory and call the API separately for each file for every prompt
    guidelines_doc_data= []
    counter1=0
    sum_of_characters=0

    for filename in os.listdir(guideline_dir):
        if filename.endswith('.txt'):
            with open(os.path.join(guideline_dir, filename), 'r') as file:
                guideline_text = file.read()
            
            text_splitter = RecursiveCharacterTextSplitter(
                        chunk_size=max_chunk_size,
                        chunk_overlap=200,
                        separators=["\n", ".", " "]
                    )

            # Iterate over each chunk amd merge adjacent chunks smaller until we reach the max chunk size, then create a new chunk and repeat the process
            temp_chunk=""
            for chunks in guideline_text.split("\n\n"):
                if len(chunks) > max_chunk_size:
                    doc = Document(page_content=temp_chunk, metadata={"source": filename, "chunk_id": counter1, "char_count": len(temp_chunk)})
                    guidelines_doc_data.append(doc)
                    counter1+=1
                    sum_of_characters+=len(temp_chunk)

                    # If the chunk itself is larger than the max chunk size, perform RecursiveTextSplitter with 1000 character chunk size and 200 character overlap to split the chunk into smaller pieces
                    sub_chunks = text_splitter.split_text(chunks)
                    for sub_chunk in sub_chunks:
                        if len(sub_chunk) > 0:
                            doc = Document(page_content=sub_chunk, metadata={"source": filename, "chunk_id": counter1, "char_count": len(sub_chunk)})
                            guidelines_doc_data.append(doc)
                            counter1+=1
                            sum_of_characters+=len(sub_chunk)
                else:
                    # If the chunk is smaller than max_chunk_size, add it to temp_chunk until temp_chunk reaches max_chunk_size, then create a new chunk and repeat the process
                    if len(temp_chunk) + len(chunks) <= max_chunk_size:
                        temp_chunk+=chunks + "\n\n"
                    else:
                        if len(temp_chunk) > 0:
                            doc = Document(page_content=temp_chunk, metadata={"source": filename, "chunk_id": counter1, "char_count": len(temp_chunk)})
                            guidelines_doc_data.append(doc)
                            counter1+=1
                            sum_of_characters+=len(temp_chunk)
                        
                        temp_chunk = chunks + "\n\n"
                        
            # If there is any remaining temp_chunk, add it as the final chunk
            if len(temp_chunk)>0:
                doc = Document(page_content=temp_chunk, metadata={"source": filename, "chunk_id": counter1, "char_count": len(temp_chunk)})
                guidelines_doc_data.append(doc)
                counter1+=1
                sum_of_characters+=len(temp_chunk)

    # Iterate through guidelines_doc_data and merge the chunk with less than 100 characters with the next chunk.
    guidelines_doc_data1= []
    for i in range(len(guidelines_doc_data)-1):
        if guidelines_doc_data[i].metadata["char_count"] <= 100:
            if guidelines_doc_data[i+1].metadata["source"] == guidelines_doc_data[i].metadata["source"]:
                guidelines_doc_data[i+1].page_content = guidelines_doc_data[i].page_content + "\n" + guidelines_doc_data[i+1].page_content
                guidelines_doc_data[i+1].metadata["char_count"] = guidelines_doc_data[i].metadata["char_count"] + guidelines_doc_data[i+1].metadata["char_count"]
                #= guidelines_doc_data[i].page_content + "\n" + guidelines_doc_data[i+1].page_content
                #guidelines_doc_data1.append(Document(page_content=guidelines_doc_data[i].page_content + "\n" + guidelines_doc_data[i+1].page_content, metadata={"source": guidelines_doc_data[i].metadata["source"], "chunk_id": guidelines_doc_data[i+1].metadata["chunk_id"], "char_count": guidelines_doc_data[i].metadata["char_count"] + guidelines_doc_data[i+1].metadata["char_count"]}))
            else:
                guidelines_doc_data1.append(guidelines_doc_data[i])
        else:
            guidelines_doc_data1.append(guidelines_doc_data[i])
    guidelines_doc_data1.append(guidelines_doc_data[-1])

    print(f"Split {len(os.listdir(guideline_dir))} documents into {len(guidelines_doc_data1)} chunks. The mean character count per chunk is {sum_of_characters/counter1} characters.")

    save_docs_to_jsonl(guidelines_doc_data1, os.path.join(output_dir, output_file_name))

    metadata_list = [doc.metadata for doc in guidelines_doc_data1]
    metadata_df = pd.DataFrame(metadata_list)
    metadata_df.to_csv(os.path.join(output_dir, output_file_name.replace(".jsonl", "_metadata.csv")), index=False)

    return guidelines_doc_data1

In [17]:
guidelines_doc_periop_data= layout_aware_chunking(periop_guideline_dir, max_chunk_size=1000, output_file_name="periop_pubmed_guidelines_chunks_Feb22.jsonl")

Split 36 documents into 3246 chunks. The mean character count per chunk is 650.7640511964385 characters.


In [18]:
guidelines_painmg_doc_data= layout_aware_chunking(painmg_guideline_dir, max_chunk_size=1000, output_file_name="painmg_pubmed_guidelines_chunks_Feb22.jsonl")

Split 31 documents into 1963 chunks. The mean character count per chunk is 628.9649122807018 characters.


In [19]:
guidelines_pdf_doc_data= layout_aware_chunking(pdf_guideline_dir, max_chunk_size=1000, output_file_name="pdf_guidelines_chunks_Feb22.jsonl")

Split 12 documents into 753 chunks. The mean character count per chunk is 652.1830822711472 characters.
